In [1]:
import os
from typing import List

import pandas as pd


# =========================
# 1. Load Compustat data
# =========================

def load_compustat_data(path: str) -> pd.DataFrame:
    """
    Load raw Compustat data from a CSV file.
    The file is read as text first; numeric conversion happens later.
    """
    df = pd.read_csv(path, dtype=str)
    return df


# =========================
# 2. Preprocess the dataset
# =========================

def preprocess_compustat(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep relevant Compustat fields and convert them to usable formats.
    """

    # Columns to keep
    useful: List[str] = [
        "gvkey", "datadate", "fyear", "conm", "tic",

        "act", "lct", "at", "lt",
        "seq", "teq", "ceq",
        "dlc", "dltt", "che", "invt", "wcap",
        "revt", "cogs", "dp",
        "ebit", "ebitda", "xint",
        "capx", "oancf", "fincf", "ivncf", "wcapch",
        "re", "pstk", "opincar",

        "csho", "mkvalt", "prcc_f", "bkvlps",

        "nipfc",
    ]

    # Keep only columns that exist in the file
    cols_exist = [c for c in useful if c in df.columns]
    df = df[cols_exist]

    # Drop invalid firm identifiers or fiscal years
    if "gvkey" in df.columns:
        df = df.dropna(subset=["gvkey"])
    if "fyear" in df.columns:
        df = df.dropna(subset=["fyear"])

    # Convert datadate → datetime
    if "datadate" in df.columns:
        df["datadate"] = pd.to_datetime(df["datadate"], errors="coerce")

    # Convert numerical fields
    numeric_cols = [
        "act", "lct", "at", "lt",
        "seq", "teq", "ceq",
        "dlc", "dltt", "che", "invt", "wcap",
        "revt", "cogs", "dp",
        "ebit", "ebitda", "xint",
        "capx", "oancf", "fincf", "ivncf", "wcapch",
        "re", "pstk", "opincar",
        "csho", "mkvalt", "prcc_f", "bkvlps",
        "nipfc",
    ]
    numeric_cols = [c for c in numeric_cols if c in df.columns]

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Convert fiscal year to integer
    if "fyear" in df.columns:
        df["fyear"] = pd.to_numeric(df["fyear"], errors="coerce").astype("Int64")

    return df


# =========================
# 3. Compute financial KPIs
# =========================

def compute_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a selected set of key financial KPIs from Compustat data.

    KPIs computed:
    - roa
    - ebitda_margin
    - debt_ratio
    - total_debt_to_equity
    - current_ratio
    - cash_ratio
    - interest_coverage
    - cfo_margin
    - free_cash_flow
    - asset_turnover
    - sales_growth
    - asset_growth
    - book_to_market
    - price_to_book
    - working_capital_to_assets
    - retained_earnings_to_assets
    """
    d = df.copy()

    # Base building blocks

    # Net income
    if "nipfc" in d.columns:
        d["net_income"] = d["nipfc"]

    # Total debt = long-term + short-term
    total_debt = None
    if {"dltt", "dlc"}.issubset(d.columns):
        d["total_debt"] = d["dltt"] + d["dlc"]
        total_debt = d["total_debt"]

    # EBT = EBIT - interest expense
    ebt = None
    if {"ebit", "xint"}.issubset(d.columns):
        d["ebt"] = d["ebit"] - d["xint"]
        ebt = d["ebt"]

    # Generic equity measure for some ratios: seq → ceq → teq
    equity_seq = None
    for eq in ["seq", "ceq", "teq"]:
        if eq in d.columns:
            equity_seq = d[eq]
            break

    # ========= Profitability =========

    # ROA = EBT / total assets
    if ebt is not None and "at" in d.columns:
        d["roa"] = d["ebt"] / d["at"]

    # EBITDA margin = EBITDA / revenue
    if {"ebitda", "revt"}.issubset(d.columns):
        d["ebitda_margin"] = d["ebitda"] / d["revt"]

    # ========= Leverage / capital structure =========

    # Debt ratio = total liabilities / total assets
    if {"lt", "at"}.issubset(d.columns):
        d["debt_ratio"] = d["lt"] / d["at"]

    # Total debt to equity = (DLTT + DLC) / equity
    if total_debt is not None and equity_seq is not None:
        d["total_debt_to_equity"] = total_debt / equity_seq

    # ========= Liquidity =========

    # Current ratio = ACT / LCT
    if {"act", "lct"}.issubset(d.columns):
        d["current_ratio"] = d["act"] / d["lct"]

    # Cash ratio = CHE / LCT
    if {"che", "lct"}.issubset(d.columns):
        d["cash_ratio"] = d["che"] / d["lct"]

    # ========= Coverage =========

    # Interest coverage = EBIT / |XINT|
    if {"ebit", "xint"}.issubset(d.columns):
        d["interest_coverage"] = d["ebit"] / d["xint"].abs()

    # ========= Cash flow metrics =========

    # CFO margin = OANCF / revenue
    if {"oancf", "revt"}.issubset(d.columns):
        d["cfo_margin"] = d["oancf"] / d["revt"]

    # Free cash flow = OANCF - CAPX
    if {"oancf", "capx"}.issubset(d.columns):
        d["free_cash_flow"] = d["oancf"] - d["capx"]

    # ========= Activity / efficiency =========

    # Asset turnover = revenue / total assets
    if {"revt", "at"}.issubset(d.columns):
        d["asset_turnover"] = d["revt"] / d["at"]

    # ========= Growth (by gvkey across years) =========

    if {"gvkey", "fyear"}.issubset(d.columns):
        d = d.sort_values(["gvkey", "fyear"])

        # Sales growth
        if "revt" in d.columns:
            d["sales_growth"] = d.groupby("gvkey")["revt"].pct_change()

        # Asset growth
        if "at" in d.columns:
            d["asset_growth"] = d.groupby("gvkey")["at"].pct_change()

    # ========= Distress-style ratios =========

    # Working capital to assets = (ACT - LCT) / AT
    if {"act", "lct", "at"}.issubset(d.columns):
        d["working_capital_to_assets"] = (d["act"] - d["lct"]) / d["at"]

    # Retained earnings to assets = RE / AT
    if {"re", "at"}.issubset(d.columns):
        d["retained_earnings_to_assets"] = d["re"] / d["at"]

    # ========= Market-based ratios =========

    # Book-to-market using CEQ → SEQ
    equity_for_bm = None
    for eq in ["ceq", "seq"]:
        if eq in d.columns:
            equity_for_bm = d[eq]
            break
    if equity_for_bm is not None and "mkvalt" in d.columns:
        d["book_to_market"] = equity_for_bm / d["mkvalt"]

    # Price-to-book = PRCC_F / BKVLPS
    if {"prcc_f", "bkvlps"}.issubset(d.columns):
        d["price_to_book"] = d["prcc_f"] / d["bkvlps"]

    return d


# =========================
# 4. Main execution script
# =========================

if __name__ == "__main__":
    path_input = (
        "/files/financial-kpis-analysis-and-distress-prediction/"
        "data/raw/compustat_data.csv"
    )
    path_output = (
        "/files/financial-kpis-analysis-and-distress-prediction/"
        "data/processed/compustat_kpis.csv"
    )

    os.makedirs(os.path.dirname(path_output), exist_ok=True)

    df_raw = load_compustat_data(path_input)
    df_prep = preprocess_compustat(df_raw)
    df_kpis = compute_kpis(df_prep)

    df_kpis.to_csv(path_output, index=False)
    print(f"KPIs calculated and saved to: {path_output}")


/tmp/ipykernel_532/3499740755.py:196: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  d["sales_growth"] = d.groupby("gvkey")["revt"].pct_change()
/tmp/ipykernel_532/3499740755.py:200: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  d["asset_growth"] = d.groupby("gvkey")["at"].pct_change()


KPIs calculated and saved to: /files/financial-kpis-analysis-and-distress-prediction/data/processed/compustat_kpis.csv
